# ZAC-GPT-2 — Early Pre-train vs Final Model

Comparison across multiple capability dimensions between the earliest available
checkpoint and the most complete model after all training stages.

| Model | Run ID | Step | Training |
|---|---|---|---|
| **Early pre-train** | `qfkhmres` | 4,000 | ~720M tokens, FineWeb-Edu, raw text |
| **Final model** | `c9pt8niy` | 81,940 | Pre-train → Mid-train → SFT → DPO → Code RL |

Sections 1–5 use **raw text completion** on both models so the comparison is direct.
Section 6 uses the **chat format** to show capabilities the pre-train model has no concept of.

In [ ]:
import sys, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

sys.path.insert(0, '..')
from core.models import TinyGPT

N_LAYERS    = 20
DIM         = N_LAYERS * 64
N_HEADS     = max(1, (DIM + 127) // 128)
MAX_SEQ_LEN = 2048
VOCAB_SIZE  = 50262
TEMPERATURE = 0.8
TOP_K       = 40

CHECKPOINTS = {
    'early': '../logs/qfkhmres/checkpoints/checkpoint_4000.pt',
    'final': '../logs/c9pt8niy/checkpoints/checkpoint_81940.pt',
}

def get_device():
    if torch.cuda.is_available(): return torch.device('cuda')
    if hasattr(torch, 'xpu') and torch.xpu.is_available(): return torch.device('xpu')
    return torch.device('cpu')

DEVICE = get_device()
print(f'Device: {DEVICE}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({
    'bos_token': '<|beginoftext|>',
    'pad_token': '<|pad|>',
    'additional_special_tokens': ['<|user|>', '<|assistant|>', '<|system|>'],
})

BOS_ID  = tokenizer.bos_token_id
EOS_ID  = tokenizer.eos_token_id
USR_ID  = tokenizer.convert_tokens_to_ids('<|user|>')
ASST_ID = tokenizer.convert_tokens_to_ids('<|assistant|>')
SYS_ID  = tokenizer.convert_tokens_to_ids('<|system|>')

In [ ]:
def load_model(stage):
    path = CHECKPOINTS[stage]
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    state_dict = ckpt['model_state_dict']
    ckpt_vocab = state_dict['tok_emb.weight'].shape[0]

    model = TinyGPT(vocab_size=ckpt_vocab, dim=DIM, n_layers=N_LAYERS,
                    n_heads=N_HEADS, max_seq_len=MAX_SEQ_LEN)
    model.load_state_dict(state_dict)

    if ckpt_vocab < VOCAB_SIZE:
        old_tok  = model.tok_emb.weight.data.clone()
        old_head = model.head.weight.data.clone()
        model.tok_emb = nn.Embedding(VOCAB_SIZE, DIM)
        model.head    = nn.Linear(DIM, VOCAB_SIZE, bias=False)
        model.tok_emb.weight.data[:ckpt_vocab] = old_tok
        model.head.weight.data[:ckpt_vocab]    = old_head

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'[{stage}] {path.split("/")[-1]} | {n_params:.0f}M params | vocab={ckpt_vocab}')
    return model.to(DEVICE).eval()


def unload(model):
    del model
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    if DEVICE.type == 'xpu':  torch.xpu.empty_cache()


@torch.no_grad()
def generate(model, prompt_ids, stop_ids=frozenset(), max_new_tokens=100):
    idx = torch.tensor([prompt_ids], device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(idx[:, -MAX_SEQ_LEN:])[:, -1, :] / TEMPERATURE
        v, _   = torch.topk(logits, min(TOP_K, logits.size(-1)))
        logits[logits < v[:, [-1]]] = float('-inf')
        next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, next_id], dim=1)
        if next_id.item() in stop_ids:
            break
    out = idx[0, len(prompt_ids):].tolist()
    if out and out[-1] in stop_ids:
        out = out[:-1]
    return out


def complete(model, prompt, max_new_tokens=80):
    """Raw text completion — works for both models."""
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    out = generate(model, ids, stop_ids={EOS_ID}, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out, skip_special_tokens=True).strip()


def chat(model, turns, system='You are a concise and helpful assistant.', max_new_tokens=150):
    """Chat format — meaningful only for the final model."""
    tokens = [BOS_ID]
    if system:
        tokens += [SYS_ID] + tokenizer.encode(system, add_special_tokens=False)
    for turn in turns:
        if turn['role'] == 'user':
            if tokens[-1] != USR_ID:
                tokens.append(USR_ID)
            tokens += tokenizer.encode(turn['content'], add_special_tokens=False)
        elif turn['role'] == 'assistant':
            tokens += [ASST_ID] + tokenizer.encode(turn['content'], add_special_tokens=False) + [USR_ID]
    tokens.append(ASST_ID)
    out = generate(model, tokens, stop_ids={EOS_ID, USR_ID}, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out, skip_special_tokens=True).strip()


def compare(prompt, max_new_tokens=80):
    """Load both models, run the same raw completion prompt, print side by side."""
    model = load_model('early')
    r_early = complete(model, prompt, max_new_tokens)
    unload(model)
    model = load_model('final')
    r_final = complete(model, prompt, max_new_tokens)
    unload(model)
    print(f'Prompt: "{prompt}"')
    print(f'  EARLY : {r_early}')
    print(f'  FINAL : {r_final}')

---
## 1. World Knowledge

From NOTES.md — these exact prompts were used to track progress during pre-training.
At step 4,000 the model outputs the most common token patterns.
By step 38,200 it could already retrieve Mount Everest from the FineWeb-Edu distribution.

In [ ]:
compare("The tallest mountain in the world is")
print()
compare("The capital of France is")
print()
compare("The speed of light is approximately")

---
## 2. Coherence & Grammar

At step 4,000 the model is still dominated by high-frequency tokens
(function words, newlines). It has not yet learned sentence structure.
The notes describe this stage as: *random words → modal words → sentence fragments*.

In [ ]:
compare("Once upon a time there was a young girl who", max_new_tokens=60)
print()
compare("Scientists have recently discovered that", max_new_tokens=60)

---
## 3. Repetition

A failure mode documented in the notes across both mid-training and SFT:
the model repeats phrases or generates degenerate loops. This is especially
visible in early pre-training where the model latches onto high-probability tokens.
DPO training specifically targeted this.

In [ ]:
compare("The most important thing to remember is", max_new_tokens=80)
print()
compare("In conclusion, we can say that", max_new_tokens=80)

---
## 4. Arithmetic & Simple Reasoning

The final model had SFT on GSM8K (math reasoning) and a small amount of
Code RL on calculator tasks, so it has been exposed to numeric reasoning.
The early model has no concept of arithmetic — numbers are just tokens.

In [ ]:
compare("If John has 10 apples and gives 3 to Mary, John now has")
print()
compare("The sum of all integers from 1 to 10 is")
print()
compare("A car travels at 60 miles per hour. In 2 hours it will have traveled")

---
## 5. Code Completion

FineWeb-Edu contains a lot of code and technical content, so both models
have been exposed to Python syntax during pre-training. The question is
whether the early model has internalized enough structure to complete
a function signature coherently.

In [ ]:
compare("def add(a, b):\n    return")
print()
compare("def fibonacci(n):\n    if n <= 1:\n        return n\n    return")
print()
compare("# Sort a list in ascending order\nmy_list = [3, 1, 4, 1, 5]\n")

---
## 6. Chat Capabilities (final model only)

The early pre-train model has no concept of a conversation format — it was
trained purely on next-token prediction over raw text. The final model went
through mid-training (special tokens), SFT (assistant-only loss mask),
DPO (preference pairs), and Code RL.

These prompts test capabilities that simply don't exist in the pre-train model:
instruction following, multi-turn context, topic switching, and structured output.

In [ ]:
# Instruction following
model = load_model('final')
r1 = chat(model, [{'role': 'user', 'content': 'List three planets in our solar system.'}])
unload(model)
print('Instruction following:')
print(f'  {r1}')
print()

# Multi-turn topic switch — kept as one block since turns build on each other
model = load_model('final')
topic_switch = [
    {'role': 'user',      'content': 'What is the capital of France?'},
    {'role': 'assistant', 'content': 'The capital of France is Paris.'},
    {'role': 'user',      'content': 'What is the capital of Germany?'},
    {'role': 'assistant', 'content': 'The capital of Germany is Berlin.'},
    {'role': 'user',      'content': 'Write a short poem about the ocean.'},
]
r2 = chat(model, topic_switch, max_new_tokens=120)
unload(model)
print('Multi-turn topic switch:')
print(f'  {r2}')
print()

# Calculator task
model = load_model('final')
r3 = chat(model, [{'role': 'user', 'content': 'What is 127 multiplied by 43? Write Python code to compute it.'}], max_new_tokens=200)
unload(model)
print('Calculator task:')
print(f'  {r3}')
print(f'  [code block present: {"```python" in r3}]')